# Embedding comparison: z-scored versus composition-corrected markers

Two UMAP embeddings of the same 354,435 cells, built to be compared directly.

### The problem this addresses

A large fraction of variance in this panel is a **global chromatin-abundance axis** — all histone
modifications rising and falling together within a cell. It is not a loading artefact (correlation
with the excluded H3/H3.3/H4 controls is only $-0.08$) and it is strongly proliferation-linked
($r \approx 0.71$–$0.88$ with KI67 across lines), so it is real biology. But it is *quantity*, not
*state*, and it can dominate an embedding.

**Z-scoring does not remove it.** Standardising each marker across cells is a **column** operation;
the global axis is a **row** effect — a coordinated shift across markers within a cell. A column
operation cannot remove a row effect, and this notebook measures exactly how much survives.

### The two representations

| | operation | removes the axis? |
|---|---|---|
| **A. z-scored** | per-marker standardisation across cells | no |
| **B. composition-corrected** | A, then centre each cell on its own mean histone signal | yes, exactly |

Representation B is a centred-log-ratio (CLR) style transform applied to the histone sub-panel: it
asks *which* modifications dominate a cell's chromatin rather than *how much* chromatin it carries.

### Which to use

Both, for different questions. B discards a genuinely biological signal, so it is not a strict
improvement — it is a different view:

- **A** shows proliferation-linked chromatin structure. Use it to see that axis.
- **B** shows what remains once chromatin quantity is accounted for. Use it to ask which marks
  dominate, or when the axis would otherwise crowd out weaker structure.

## 0. Setup and parameters

In [ ]:
import sys, json
from pathlib import Path

CE_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/CyEmbed"
CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
for p in (CE_PATH, CT_PATH):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np, pandas as pd, anndata as ad
import matplotlib.pyplot as plt, seaborn as sns
import umap
from sklearn.neighbors import NearestNeighbors

from cytofstandard import Project
from CyEmbed.data import extract_matrix, fit_scaler, preprocess_array

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11})

BASE = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
PLOTS = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)
OUT = BASE / "outputs/embeddings"; OUT.mkdir(parents=True, exist_ok=True)

LINES = ["MDAMB468", "HCC70", "SUM149", "HCC1937", "MCF7"]
LINE_DISP = {"MDAMB468": "MDA-MB-468", "HCC70": "HCC70", "SUM149": "SUM149",
             "HCC1937": "HCC1937", "MCF7": "MCF7"}
LINE_ORDER = ["HCC1937", "HCC70", "MCF7", "MDA-MB-468", "SUM149"]

# ---- parameters -------------------------------------------------------------------------------
SEED          = 42
UMAP_FIT_N    = 60_000     # cells used to FIT the manifold (stratified by line)
UMAP_CHUNK    = 50_000     # transform batch size
N_NEIGHBORS   = 30
MIN_DIST      = 0.3
DETERMINISTIC = True       # True -> reproducible but single-threaded; False -> parallel, stochastic
LAYER         = "norm_divide"
print(f"deterministic={DETERMINISTIC} (set False for a ~5-10x faster but non-reproducible layout)")

## 1. Data and the shared scaler

One z-score scaler is fitted across all five lines on a per-line balanced subsample, so the lines
share a common marker space.

In [ ]:
adatas = {}
for line in LINES:
    a = Project.load(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{line}_NormCompare") \
               .get_run(line).read_adata()
    bio = [m for m in a.var_names if m not in ["H3", "H3.3", "H4"]]
    sub = ad.AnnData(X=a[:, bio].layers[LAYER].copy(),
                     obs=a.obs[["cell_uuid"]].copy(), var=a[:, bio].var.copy())
    sub.obs["cell_line"] = LINE_DISP[line]
    adatas[line] = sub

combined = ad.concat([adatas[l] for l in LINES], join="outer")
bundle = extract_matrix(adata=combined, layer=None, sample_col="cell_line")
SCALER, _ = fit_scaler(bundle.X, mode="zscore",
                       sample_ids=bundle.sample_ids, balanced_max_per_sample=5000)

X_Z = preprocess_array(bundle.X, SCALER).astype(np.float32)   # representation A
MARKERS = list(combined.var_names)
cell_line = combined.obs["cell_line"].to_numpy()

HIST = [m for m in MARKERS if m.startswith(("H2A", "H3", "H4")) and m != "H3S28p"]
hist_idx = [MARKERS.index(m) for m in HIST]

print(f"{X_Z.shape[0]:,} cells x {X_Z.shape[1]} markers")
print(f"{len(HIST)} histone-PTM channels (H3S28p excluded: mitosis-specific, not an abundance mark)")
print(pd.Series(cell_line).value_counts().reindex(LINE_ORDER).to_string())

## 2. The two representations, and how much axis each carries

In [ ]:
# global chromatin score: mean of the histone-PTM channels
global_chrom = X_Z[:, hist_idx].mean(1)

# representation B: centre each cell on its own mean histone signal (CLR over the histone panel)
X_C = X_Z.copy()
X_C[:, hist_idx] = X_Z[:, hist_idx] - X_Z[:, hist_idx].mean(1, keepdims=True)

REPS = {"A. z-scored": X_Z, "B. composition-corrected": X_C}

u = np.zeros(len(MARKERS), dtype=np.float32); u[hist_idx] = 1.0
u /= np.linalg.norm(u)

print("=== Variance carried by the global chromatin direction ===")
for name, R in REPS.items():
    share = 100 * (R @ u).var() / R.var(axis=0).sum()
    print(f"  {name:<26} {share:5.2f}%")

ki = X_Z[:, MARKERS.index("KI67")]
print(f"\nglobal chromatin score vs KI67: r = {np.corrcoef(global_chrom, ki)[0, 1]:+.3f}")
print("-> the axis is proliferation-linked, i.e. real biology, not merely technical.")
print("   Removing it is a deliberate choice about which question to ask, not a cleanup step.")

## 3. Embeddings

Both use the same protocol: fit the manifold on a line-stratified subsample, then project every
cell. Cells used for the fit keep their exact fitted coordinates.

In [ ]:
def stratified_idx(labels, fit_n, seed=SEED):
    labels = np.asarray(labels); rng = np.random.default_rng(seed); n = len(labels)
    if fit_n >= n:
        return np.arange(n)
    parts = []
    for lab in np.unique(labels):
        pool = np.flatnonzero(labels == lab)
        take = max(1, int(round(fit_n * len(pool) / n)))
        parts.append(rng.choice(pool, size=min(take, len(pool)), replace=False))
    return np.sort(np.concatenate(parts))


def embed(rep, labels, name=""):
    rep = np.ascontiguousarray(np.asarray(rep, np.float32))
    fit = stratified_idx(labels, UMAP_FIT_N)
    kw = dict(n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST, verbose=False)
    kw.update(dict(random_state=SEED) if DETERMINISTIC else dict(n_jobs=-1))
    red = umap.UMAP(**kw).fit(rep[fit])
    out = np.empty((len(rep), 2), np.float32)
    out[fit] = red.embedding_
    rest = np.setdiff1d(np.arange(len(rep)), fit)
    for s in range(0, len(rest), UMAP_CHUNK):
        b = rest[s:s + UMAP_CHUNK]
        out[b] = red.transform(rep[b])
    print(f"  [{name}] fit on {len(fit):,}, projected {len(rep):,}", flush=True)
    return out


def cache_path(name):
    return OUT / f"umap_{'zscored' if name.startswith('A') else 'composition'}.npy"


# Embeddings are cached: recomputing UMAP is the slow step and nothing downstream changes it.
# Delete the .npy files (or set FORCE_RECOMPUTE) to rebuild.
FORCE_RECOMPUTE = False

EMB = {}
for name, R in REPS.items():
    f = cache_path(name)
    if f.exists() and not FORCE_RECOMPUTE:
        EMB[name] = np.load(f)
        print(f"  [{name}] loaded from cache {f.name}")
    else:
        EMB[name] = embed(R, cell_line, name)
        np.save(f, EMB[name])
        print(f"  [{name}] computed and saved -> {f.name}")

## 4. Quantitative comparison

The question is *how much of the global chromatin axis is still readable from the embedding*. A
correlation between the axis and a single UMAP coordinate answers this badly: UMAP's axes are
arbitrarily rotated, so a diagonal gradient is split between them and understated, and Pearson
correlation sees only linear structure. An in-sample grid statistic fails the other way, inflating
with grid resolution.

Regression with held-out evaluation avoids both. Two models are fitted from the 2-D coordinates to
the per-cell chromatin score:

* **Linear, random split** --- rotation-invariant by construction, and measures *directional*
  structure: can a gradient be read off the plot?
* **Gradient boosting, spatially blocked split** --- entire tiles of the embedding are held out, so
  test points have no training neighbours nearby. This matters: with 354,435 cells in two dimensions
  a random split leaves near-duplicates in the training set, and a flexible model then scores highly
  by interpolation rather than by using spatial structure. Blocking measures what position genuinely
  predicts.

Neighbourhood purity is reported alongside as a model-free check, and a permuted target gives the
floor.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score


def purity(rep, labels, k=30, sample=15000, seed=SEED):
    rng = np.random.default_rng(seed); labels = np.asarray(labels)
    i = rng.choice(len(rep), min(sample, len(rep)), replace=False)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(rep)
    _, ind = nn.kneighbors(rep[i])
    return float((labels[ind[:, 1:]] == labels[i][:, None]).mean())


def blocked_split(e, n_tiles=8, test_frac=0.3, seed=SEED):
    """Hold out whole tiles of the embedding so test points have no nearby training points."""
    r = np.random.default_rng(seed)
    bx = np.clip(np.digitize(e[:, 0], np.linspace(e[:, 0].min(), e[:, 0].max(), n_tiles + 1)[1:-1]),
                 0, n_tiles - 1)
    by = np.clip(np.digitize(e[:, 1], np.linspace(e[:, 1].min(), e[:, 1].max(), n_tiles + 1)[1:-1]),
                 0, n_tiles - 1)
    tile = bx * n_tiles + by
    uniq = np.unique(tile)
    test_tiles = r.choice(uniq, size=max(1, int(test_frac * len(uniq))), replace=False)
    te = np.isin(tile, test_tiles)
    return ~te, te


rng_split = np.random.default_rng(SEED)
rand_mask = rng_split.random(len(global_chrom)) < 0.7
chrom_q = pd.qcut(global_chrom, 4, labels=False)

rows = []
for name, R in REPS.items():
    e = EMB[name].astype(np.float64)
    # directional structure: linear, random split, rotation-invariant
    lin = LinearRegression().fit(e[rand_mask], global_chrom[rand_mask])
    r2_lin = r2_score(global_chrom[~rand_mask], lin.predict(e[~rand_mask]))
    # total recoverable: flexible model, spatially blocked
    tr, te = blocked_split(e)
    gbm = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(e[tr], global_chrom[tr])
    r2_blk = r2_score(global_chrom[te], gbm.predict(e[te]))
    rows.append({
        "representation": name,
        "axis_var_%": 100 * (R @ u).var() / R.var(axis=0).sum(),
        "R2 linear (directional)": r2_lin,
        "R2 blocked GBM (total)": r2_blk,
        "NN purity: chromatin quartile": purity(R, chrom_q),
        "NN purity: cell line": purity(R, cell_line),
    })
cmp_df = pd.DataFrame(rows).set_index("representation")
print(cmp_df.round(3).to_string())

# floor: the same model on a permuted target should explain nothing
e0 = EMB[list(REPS)[0]].astype(np.float64)
y_perm = rng_split.permutation(global_chrom)
gbm0 = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(e0[rand_mask], y_perm[rand_mask])
print(f"\nfloor check (permuted target): R^2 = {r2_score(y_perm[~rand_mask], gbm0.predict(e0[~rand_mask])):+.4f}")

print("\nReading:")
print("  axis_var_%              -> composition correction removes the axis from the representation")
print("  R2 linear               -> can a chromatin gradient be READ OFF the plot? (rotation-invariant)")
print("  R2 blocked GBM          -> what position genuinely predicts, without interpolation from")
print("                             near-duplicate neighbours; the honest 'residual structure' number")
print("  NN purity: quartile     -> model-free companion to the above")
print("  NN purity: cell line    -> control; should be ~unchanged by the correction")

## 5. Side-by-side overview

In [ ]:
def panel(ax, e, values, categorical, title, cmap="magma", s=0.7):
    if categorical:
        for c, col in zip(LINE_ORDER, sns.color_palette("tab10", len(LINE_ORDER))):
            m = values == c
            ax.scatter(e[m, 0], e[m, 1], s=s, color=col, alpha=0.45, rasterized=True, label=c)
    else:
        lo, hi = np.quantile(values, 0.02), np.quantile(values, 0.98)
        ax.scatter(e[:, 0], e[:, 1], s=s, c=values, cmap=cmap, alpha=0.55,
                   vmin=lo, vmax=hi, rasterized=True)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


overlays = [("global chromatin score", global_chrom, False, "magma"),
            ("KI67", X_Z[:, MARKERS.index("KI67")], False, "viridis"),
            ("cell line", cell_line, True, None)]

fig, axes = plt.subplots(len(overlays), 2, figsize=(13, 6.2 * len(overlays)))
for r, (ttl, vals, cat, cmap) in enumerate(overlays):
    for c, name in enumerate(REPS):
        panel(axes[r, c], EMB[name], vals, cat, f"{name}\n{ttl}", cmap=cmap or "magma")
axes[2, 0].legend(markerscale=14, fontsize=9, frameon=False, loc="best")
fig.suptitle("Same cells, two preprocessing choices", fontsize=15, fontweight="bold", y=0.999)
plt.tight_layout()
plt.savefig(PLOTS / "EmbedCompare_Overview.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Markers on both embeddings

The histone channels are the informative comparison: in representation A they should all look
similar to one another (they are riding the same axis); in B they should differentiate.

In [ ]:
def grid(names, values_fn, fname, suptitle, cmap="magma"):
    n = len(names); ncols = 6; nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols * 2, figsize=(ncols * 2 * 2.5, nrows * 2.7))
    axf = np.atleast_1d(axes).reshape(nrows, ncols * 2)
    for i, nm in enumerate(names):
        r_, c_ = divmod(i, ncols)
        for j, rep in enumerate(REPS):
            panel(axf[r_, c_ * 2 + j], EMB[rep], values_fn(rep, nm), False,
                  f"{nm}\n{'A' if j == 0 else 'B'}", cmap=cmap, s=0.5)
    for ax in axf.ravel()[n * 2:]:
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=15, fontweight="bold", y=1.002)
    plt.tight_layout()
    plt.savefig(PLOTS / fname, dpi=130, bbox_inches="tight")
    plt.show()


grid(HIST, lambda rep, nm: REPS[rep][:, MARKERS.index(nm)],
     "EmbedCompare_HistoneMarks.png",
     "Histone modifications: A (z-scored) vs B (composition-corrected), paired")

In [ ]:
LINEAGE = [m for m in MARKERS if m not in HIST]
grid(LINEAGE, lambda rep, nm: REPS[rep][:, MARKERS.index(nm)],
     "EmbedCompare_LineageMarkers.png",
     "Lineage and state markers: A vs B, paired (these are identical in both reps)")

## 7. Summary

In [ ]:
print("=" * 74)
print("Embedding comparison summary")
print("=" * 74)
print(f"\ncells {X_Z.shape[0]:,} | markers {len(MARKERS)} | histone channels {len(HIST)}")
print(f"UMAP: fit on {UMAP_FIT_N:,} stratified cells, n_neighbors={N_NEIGHBORS}, "
      f"min_dist={MIN_DIST}, deterministic={DETERMINISTIC}\n")
print(cmp_df.round(3).to_string())
a, b = cmp_df.index[0], cmp_df.index[1]
print(f"\nglobal chromatin axis: {cmp_df.loc[a,'axis_var_%']:.1f}% of variance -> "
      f"{cmp_df.loc[b,'axis_var_%']:.1f}%")
print(f"directional (linear R2):     {cmp_df.loc[a,'R2 linear (directional)']:.3f} -> "
      f"{cmp_df.loc[b,'R2 linear (directional)']:.3f}   (gradient readable off the plot?)")
print(f"total recoverable (blocked): {cmp_df.loc[a,'R2 blocked GBM (total)']:.3f} -> "
      f"{cmp_df.loc[b,'R2 blocked GBM (total)']:.3f}   (survives as local patchiness)")
print(f"cell-line purity: {cmp_df.loc[a,'NN purity: cell line']:.3f} -> "
      f"{cmp_df.loc[b,'NN purity: cell line']:.3f}  (should be ~unchanged)")
print("\nUse A to see proliferation-linked chromatin structure;")
print("use B to ask which modifications dominate once quantity is accounted for.")
print(f"\nembeddings saved to {OUT}")
print("=" * 74)